In [0]:
import pyspark.sql.functions as F

catalog = "visagio_atividade_v_energy_360"
gold_schema = f"{catalog}.gold"
sqlite_path = "/tmp/v_energy.db"

dim_unidade = spark.table(f"{gold_schema}.dim_unidade_consumidora")
fato_medicao = spark.table(f"{gold_schema}.fato_medicao")

test_results = []

def register_test(test_name, violations_count, severity="CRITICO", detail=""):
    passed = violations_count == 0
    status = "PASS" if passed else ("FAIL" if severity == "CRITICO" else "WARN")
    test_results.append(
        {"teste": test_name, "status": status, "severidade": severity,
         "violacoes": violations_count, "detalhe": detail}
    )
    print(f"[{status}] {test_name} -> {violations_count} violações {detail}")

In [0]:
total_dim_rows = dim_unidade.count()

# T1: chave primária única
duplicated_keys = (
    dim_unidade.groupBy("id_uc").count().filter("count > 1").count()
)
register_test("dim_uc: id_uc único (zero duplicatas)", duplicated_keys)

# T2: chave primária não nula
register_test(
    "dim_uc: id_uc não nulo",
    dim_unidade.filter(F.col("id_uc").isNull()).count(),
)

# T3: consistência da flag de GD
register_test(
    "dim_uc: possui_gd=false exige potencia_gd_kw = 0",
    dim_unidade.filter(
        (F.col("possui_gd") == False) & (F.coalesce(F.col("potencia_gd_kw"), F.lit(0.0)) != 0.0)
    ).count(),
)
register_test(
    "dim_uc: possui_gd=true exige potencia_gd_kw > 0",
    dim_unidade.filter(
        (F.col("possui_gd") == True) & (F.coalesce(F.col("potencia_gd_kw"), F.lit(0.0)) <= 0.0)
    ).count(),
)

# T4: valores de domínio
register_test(
    "dim_uc: uf = 'PE'",
    dim_unidade.filter(F.col("uf") != "PE").count(),
    severity="AVISO",
)
register_test(
    "dim_uc: status em (ATIVA, INATIVA)",
    dim_unidade.filter(~F.col("status").isin("ATIVA", "INATIVA")).count(),
    severity="AVISO",
)
register_test(
    "dim_uc: subgrupo_tarifario em (B1, B2, B3, A4)",
    dim_unidade.filter(~F.col("subgrupo_tarifario").isin("B1", "B2", "B3", "A4")).count(),
    severity="AVISO",
)

# T5: completude de campos que o portal usa em filtros
for filter_column in ["bairro", "municipio", "alimentador", "classe_consumo"]:
    register_test(
        f"dim_uc: {filter_column} preenchido",
        dim_unidade.filter(F.col(filter_column).isNull() | (F.trim(F.col(filter_column)) == "")).count(),
        severity="AVISO",
    )
# T6: integridade de codificação textual (zero bairros com caracteres corrompidos / mojibake)
invalid_bairros_gold = dim_unidade.filter(F.col("bairro").rlike(r"[^ -~]")).count()
register_test("dim_uc: bairros sem caracteres corrompidos (ASCII puro)", invalid_bairros_gold)


[PASS] dim_uc: id_uc único (zero duplicatas) -> 0 violações 
[PASS] dim_uc: id_uc não nulo -> 0 violações 
[PASS] dim_uc: possui_gd=false exige potencia_gd_kw = 0 -> 0 violações 
[PASS] dim_uc: possui_gd=true exige potencia_gd_kw > 0 -> 0 violações 
[PASS] dim_uc: uf = 'PE' -> 0 violações 
[PASS] dim_uc: status em (ATIVA, INATIVA) -> 0 violações 
[PASS] dim_uc: subgrupo_tarifario em (B1, B2, B3, A4) -> 0 violações 
[PASS] dim_uc: bairro preenchido -> 0 violações 
[PASS] dim_uc: municipio preenchido -> 0 violações 
[PASS] dim_uc: alimentador preenchido -> 0 violações 
[PASS] dim_uc: classe_consumo preenchido -> 0 violações 


In [0]:
# T6: chave primária única
register_test(
    "fato: id_medicao único",
    fato_medicao.groupBy("id_medicao").count().filter("count > 1").count(),
)

# T7: unicidade de (id_uc, competencia), a regra da dedup canônica
register_test(
    "fato: (id_uc, competencia) único",
    fato_medicao.groupBy("id_uc", "competencia").count().filter("count > 1").count(),
)

# T8: sem consumo negativo (exigência da Fatia)
register_test(
    "fato: consumo_kwh >= 0",
    fato_medicao.filter(F.col("consumo_kwh") < 0).count(),
)

# T9: integridade referencial, 100% dos id_uc da fato existem na dimensão
orphan_measurements = fato_medicao.join(dim_unidade.select("id_uc"), "id_uc", "left_anti").count()
register_test("fato: 100% dos id_uc existem na dim_unidade_consumidora", orphan_measurements)

# T10: formato da competência
register_test(
    "fato: competencia no formato YYYY-MM",
    fato_medicao.filter(~F.col("competencia").rlike(r"^\d{4}-(0[1-9]|1[0-2])$")).count(),
)

# T11: dias de faturamento plausíveis
register_test(
    "fato: dias_faturamento entre 1 e 62",
    fato_medicao.filter(
        F.col("dias_faturamento").isNull()
        | (F.col("dias_faturamento") < 1)
        | (F.col("dias_faturamento") > 62)
    ).count(),
    severity="AVISO",
)

# T12: consumo nulo (hoje deve acusar o problema da troca de medidor, item 3)
null_consumption_rows = fato_medicao.filter(F.col("consumo_kwh").isNull()).count()
total_fact_rows = fato_medicao.count()
register_test(
    "fato: consumo_kwh não nulo",
    null_consumption_rows,
    severity="AVISO",
    detail=f"({null_consumption_rows / total_fact_rows:.2%} das linhas)",
)

# T13: média diária coerente com consumo / dias
register_test(
    "fato: media_diaria_kwh = consumo_kwh / dias_faturamento (tolerância 0.01)",
    fato_medicao.filter(
        F.col("consumo_kwh").isNotNull()
        & (F.abs(F.col("media_diaria_kwh") - F.col("consumo_kwh") / F.col("dias_faturamento")) > 0.01)
    ).count(),
)

# T14: reconciliação leitura_atual - leitura_anterior vs consumo_kwh (item 10 da revisão)
register_test(
    "fato: consumo_kwh = leitura_atual - leitura_anterior (tolerância 1 kWh, exceto rollover)",
    fato_medicao.filter(
        F.col("consumo_kwh").isNotNull()
        & (F.col("leitura_atual") >= F.col("leitura_anterior"))
        & (F.abs(F.col("consumo_kwh") - (F.col("leitura_atual") - F.col("leitura_anterior"))) > 1.0)
    ).count(),
    severity="AVISO",
)

[PASS] fato: id_medicao único -> 0 violações 
[PASS] fato: (id_uc, competencia) único -> 0 violações 
[PASS] fato: consumo_kwh >= 0 -> 0 violações 
[PASS] fato: 100% dos id_uc existem na dim_unidade_consumidora -> 0 violações 
[PASS] fato: competencia no formato YYYY-MM -> 0 violações 
[PASS] fato: dias_faturamento entre 1 e 62 -> 0 violações 
[WARN] fato: consumo_kwh não nulo -> 6329 violações (0.29% das linhas)
[PASS] fato: media_diaria_kwh = consumo_kwh / dias_faturamento (tolerância 0.01) -> 0 violações 
[PASS] fato: consumo_kwh = leitura_atual - leitura_anterior (tolerância 1 kWh, exceto rollover) -> 0 violações 


In [0]:
import os
import shutil
import sqlite3

volume_db_path = f"/Volumes/{catalog}/gold/exports/v_energy.db"
local_copy_path = "/tmp/v_energy_check.db"

if not os.path.exists(volume_db_path):
    raise FileNotFoundError(
        f"{volume_db_path} não existe. Rode a Cell 7 do Silver_to_Gold antes dos testes."
    )

# Copia para o disco local: SQLite com WAL não funciona bem direto no Volume
shutil.copy(volume_db_path, local_copy_path)
connection = sqlite3.connect(local_copy_path)

sqlite_dim_rows = connection.execute("SELECT COUNT(*) FROM dim_unidade_consumidora").fetchone()[0]
sqlite_fact_rows = connection.execute("SELECT COUNT(*) FROM fato_medicao").fetchone()[0]

register_test("sqlite: linhas da dim = Gold", abs(sqlite_dim_rows - total_dim_rows))
register_test("sqlite: linhas da fato = Gold", abs(sqlite_fact_rows - total_fact_rows))
register_test(
    "sqlite: PRAGMA foreign_key_check sem violações",
    len(connection.execute("PRAGMA foreign_key_check;").fetchall()),
)
register_test(
    "sqlite: PRAGMA integrity_check = ok",
    0 if connection.execute("PRAGMA integrity_check;").fetchone()[0] == "ok" else 1,
)
register_test(
    "sqlite: journal_mode = wal",
    0 if connection.execute("PRAGMA journal_mode;").fetchone()[0].lower() == "wal" else 1,
)

index_names = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='index'")}
required_indexes = {"idx_dim_uc_id", "idx_dim_uc_bairro", "idx_dim_uc_mun", "idx_fato_med_uc", "idx_fato_med_comp"}
register_test("sqlite: 5 índices analíticos presentes", len(required_indexes - index_names))

dim_tempo = spark.table(f"{gold_schema}.dim_tempo")
anomalia_df = spark.table(f"{gold_schema}.fato_anomalia_leitura")

register_test("sqlite: linhas da dim_tempo = Gold",
    abs(connection.execute("SELECT COUNT(*) FROM dim_tempo").fetchone()[0] - dim_tempo.count()))
register_test("sqlite: linhas da fato_anomalia_leitura = Gold",
    abs(connection.execute("SELECT COUNT(*) FROM fato_anomalia_leitura").fetchone()[0] - anomalia_df.count()))
register_test("fato: toda competencia existe na dim_tempo",
    fato_medicao.join(dim_tempo.select("competencia"), "competencia", "left_anti").count())
register_test("anomalia: id_uc existe na dim_unidade_consumidora",
    anomalia_df.join(dim_unidade.select("id_uc"), "id_uc", "left_anti").count())
register_test("anomalia: nenhuma anomalia aparece na fato_medicao",
    anomalia_df.join(fato_medicao.select("id_uc", "competencia"), ["id_uc", "competencia"], "inner").count())


# sqlite: conferência de bairros sem caracteres corrompidos (GLOB fora da faixa ASCII)
sqlite_invalid_bairros = connection.execute(
    "SELECT COUNT(*) FROM dim_unidade_consumidora WHERE bairro GLOB '*[^ -~]*'"
).fetchone()[0]
register_test("sqlite: zero bairros com caracteres corrompidos (GLOB)", sqlite_invalid_bairros)

connection.close()

[PASS] sqlite: linhas da dim = Gold -> 0 violações 
[PASS] sqlite: linhas da fato = Gold -> 0 violações 
[PASS] sqlite: PRAGMA foreign_key_check sem violações -> 0 violações 
[PASS] sqlite: PRAGMA integrity_check = ok -> 0 violações 
[PASS] sqlite: journal_mode = wal -> 0 violações 
[PASS] sqlite: 5 índices analíticos presentes -> 0 violações 
[PASS] sqlite: linhas da dim_tempo = Gold -> 0 violações 
[PASS] sqlite: linhas da fato_anomalia_leitura = Gold -> 0 violações 
[PASS] fato: toda competencia existe na dim_tempo -> 0 violações 
[PASS] anomalia: id_uc existe na dim_unidade_consumidora -> 0 violações 
[PASS] anomalia: nenhuma anomalia aparece na fato_medicao -> 0 violações 


In [0]:
difference_df = (
    fato_medicao
    .filter(F.col("consumo_kwh").isNotNull()
            & F.col("leitura_atual").isNotNull()
            & F.col("leitura_anterior").isNotNull()
            & (F.col("leitura_atual") >= F.col("leitura_anterior")))
    .withColumn("diferenca", F.col("consumo_kwh") - (F.col("leitura_atual") - F.col("leitura_anterior")))
    .withColumn("diferenca_abs", F.abs("diferenca"))
    .withColumn("razao", F.col("consumo_kwh") / (F.col("leitura_atual") - F.col("leitura_anterior")))
)

# (a) Magnitude da divergência
display(difference_df.filter("diferenca_abs > 0.01").selectExpr(
    "count(*) as linhas",
    "round(min(diferenca),2) as min", "round(percentile_approx(diferenca, 0.5),2) as mediana",
    "round(max(diferenca),2) as max", "round(avg(diferenca_abs),2) as media_abs"))

# (b) Concentra em REAL ou ESTIMADA?
display(difference_df.groupBy("tipo_leitura").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("diferenca_abs") > 0.01, 1).otherwise(0)).alias("divergentes")))

# (c) Faixas de divergência
display(difference_df.filter("diferenca_abs > 0.01").groupBy(
    F.when(F.col("diferenca_abs") <= 1, "<= 1 kWh (arredondamento)")
     .when(F.col("diferenca_abs") <= 10, "1-10 kWh")
     .when(F.col("diferenca_abs") <= 100, "10-100 kWh")
     .otherwise("> 100 kWh").alias("faixa")).count())

# (d) Por ano: é sistemático ou pontual?
display(difference_df.groupBy(F.substring("competencia", 1, 4).alias("ano")).agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("diferenca_abs") > 0.01, 1).otherwise(0)).alias("divergentes")).orderBy("ano"))

# (e) 15 exemplos para olhar a olho nu
display(difference_df.filter("diferenca_abs > 0.01").select(
    "id_uc", "competencia", "tipo_leitura", "leitura_anterior", "leitura_atual",
    "consumo_kwh", "diferenca", "razao").limit(15))

linhas,min,mediana,max,media_abs
518042,-1.0,1.0,1.0,1.0


tipo_leitura,total,divergentes
ESTIMADA,384687,94198
REAL,1729149,423844


faixa,count
<= 1 kWh (arredondamento),518042


ano,total,divergentes
2022,263987,55605
2023,528821,131962
2024,530001,132446
2025,527805,132155
2026,263222,65874


id_uc,competencia,tipo_leitura,leitura_anterior,leitura_atual,consumo_kwh,diferenca,razao
UC100000,2022-10,ESTIMADA,8888.0,9126.0,237.0,-1.0,0.9957983193277311
UC100000,2022-12,REAL,9457.0,9894.0,438.0,1.0,1.002288329519451
UC100000,2023-02,REAL,10253.0,10675.0,421.0,-1.0,0.9976303317535545
UC100000,2023-03,ESTIMADA,10675.0,11037.0,363.0,1.0,1.0027624309392265
UC100000,2023-04,REAL,11037.0,11399.0,361.0,-1.0,0.9972375690607734
UC100000,2023-10,ESTIMADA,12412.0,12670.0,259.0,1.0,1.003875968992248
UC100000,2023-12,REAL,12968.0,13384.0,417.0,1.0,1.0024038461538463
UC100000,2024-01,ESTIMADA,13384.0,13816.0,431.0,-1.0,0.9976851851851852
UC100000,2024-05,REAL,14987.0,15260.0,274.0,1.0,1.0036630036630036
UC100000,2024-08,REAL,15671.0,15832.0,160.0,-1.0,0.9937888198757764


In [0]:
from pyspark.sql.window import Window

continuity_window = Window.partitionBy("id_uc").orderBy("competencia")
continuity_df = (
    fato_medicao
    .withColumn("leitura_atual_mes_anterior", F.lag("leitura_atual").over(continuity_window))
    .filter(F.col("leitura_atual_mes_anterior").isNotNull() & F.col("leitura_anterior").isNotNull())
)
display(continuity_df.agg(
    F.count("*").alias("pares"),
    F.sum(F.when(F.abs(F.col("leitura_anterior") - F.col("leitura_atual_mes_anterior")) > 0.01, 1).otherwise(0)).alias("quebras_de_continuidade")))

pares,quebras_de_continuidade
2075748,0


In [0]:
report_df = spark.createDataFrame(test_results)
display(report_df.orderBy("status", "teste"))

# Persiste o relatório para auditoria (serve também ao dicionário de dados)
(report_df.withColumn("executado_em", F.current_timestamp())
    .write.format("delta").mode("append")
    .saveAsTable(f"{gold_schema}.qualidade_testes_resultado"))

critical_failures = [result for result in test_results if result["status"] == "FAIL"]
warnings = [result for result in test_results if result["status"] == "WARN"]

print(f"\nResumo: {len(test_results)} testes | {len(critical_failures)} falhas críticas | {len(warnings)} avisos")

if critical_failures:
    failed_names = "; ".join(result["teste"] for result in critical_failures)
    raise AssertionError(f"Testes críticos falharam: {failed_names}")

print("Fase 5: todos os testes críticos passaram.")

detalhe,severidade,status,teste,violacoes
,CRITICO,PASS,anomalia: id_uc existe na dim_unidade_consumidora,0
,CRITICO,PASS,anomalia: nenhuma anomalia aparece na fato_medicao,0
,AVISO,PASS,dim_uc: alimentador preenchido,0
,AVISO,PASS,dim_uc: bairro preenchido,0
,AVISO,PASS,dim_uc: classe_consumo preenchido,0
,CRITICO,PASS,dim_uc: id_uc não nulo,0
,CRITICO,PASS,dim_uc: id_uc único (zero duplicatas),0
,AVISO,PASS,dim_uc: municipio preenchido,0
,CRITICO,PASS,dim_uc: possui_gd=false exige potencia_gd_kw = 0,0
,CRITICO,PASS,dim_uc: possui_gd=true exige potencia_gd_kw > 0,0



Resumo: 31 testes | 0 falhas críticas | 1 avisos
Fase 5: todos os testes críticos passaram.
